# Agente Identificador de Fraudes Bancárias
## Notebook 01 – Análise Exploratória, Preparação dos Dados e Baselines

**Universidade Presbiteriana Mackenzie – Faculdade de Computação e Informática**
Inteligência Artificial – 7º CC (Turma N) – Prof. Dr. Ivan Carlos Alcântara de Oliveira

Dataset: *Credit Card Fraud Detection* (Machine Learning Group – ULB), disponível em https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

**Como executar:** abra no Google Colab e use *Ambiente de execução → Executar tudo*. O dataset é baixado automaticamente via `kagglehub`. Se o download falhar, envie o arquivo `creditcard.csv` para a pasta do Colab e execute de novo. Ao final, o notebook gera `resultados_parciais.json` e `figuras.zip`.

In [ ]:
# =============================================================================
# Projeto : Agente Identificador de Fraudes Bancárias
# Arquivo : 01_analise_exploratoria.ipynb
# Curso   : Ciência da Computação – Universidade Presbiteriana Mackenzie (FCI)
# Disciplina: Inteligência Artificial – 7º semestre, Turma N
# Professor : Prof. Dr. Ivan Carlos Alcântara de Oliveira
#
# Integrantes:
#   Cheuk Ki Yu                  – RA 10419664 – 10419664@mackenzista.com.br
#   Gabriela Nellessen de Sousa  – RA 10441930 – 10441930@mackenzista.com.br
#   Milton Almeida Leoncio       – RA 10416764 – 10416764@mackenzista.com.br
#
# Síntese do conteúdo:
#   Carga do dataset Credit Card Fraud Detection (ULB/Kaggle); análise
#   exploratória (estrutura, valores ausentes, duplicatas, desbalanceamento,
#   distribuição de Amount e Time, separabilidade dos componentes PCA);
#   preparação dos dados (remoção de duplicatas, engenharia de atributos,
#   divisão estratificada treino/teste, padronização robusta) e modelos de
#   referência (baselines) com e sem SMOTE, avaliados por Precisão, Recall,
#   F1, MCC, ROC-AUC e PR-AUC, incluindo verificação com divisão temporal.
#
# Histórico de alterações (data | autor | descrição):
#   2026-09-24 | Cheuk Ki Yu | Criação do notebook: carga, EDA, preparação
#              |             | dos dados e baselines (LR, LR+SMOTE, RF).
# =============================================================================

In [ ]:
# Instalação das dependências (no Colab a maioria já vem instalada)
!pip install -q kagglehub imbalanced-learn

In [ ]:
import os, json, shutil, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             confusion_matrix, precision_recall_curve)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
FIG_DIR = "figuras"
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10})

RESULTADOS = {}          # tudo que vai para o artigo é registrado aqui

def salvar_fig(nome):
    caminho = os.path.join(FIG_DIR, nome)
    plt.tight_layout()
    plt.savefig(caminho, bbox_inches="tight")
    plt.show()
    print("Figura salva em:", caminho)

## 1. Carga dos dados

In [ ]:
CAMINHO_LOCAL = "creditcard.csv"
if os.path.exists(CAMINHO_LOCAL):
    df = pd.read_csv(CAMINHO_LOCAL)
else:
    import kagglehub
    pasta = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    df = pd.read_csv(os.path.join(pasta, "creditcard.csv"))

print("Dimensões:", df.shape)
df.head()

## 2. Estrutura, tipos e valores ausentes

In [ ]:
print(df.dtypes.value_counts())
ausentes = int(df.isna().sum().sum())
print("Total de valores ausentes:", ausentes)

RESULTADOS["n_linhas_original"] = int(df.shape[0])
RESULTADOS["n_colunas"] = int(df.shape[1])
RESULTADOS["valores_ausentes"] = ausentes
RESULTADOS["tempo_total_horas"] = round(float(df["Time"].max()) / 3600, 2)
df.describe().T

## 3. Duplicatas

In [ ]:
dup_mask = df.duplicated(keep="first")
n_dup = int(dup_mask.sum())
n_dup_fraude = int(df.loc[dup_mask, "Class"].sum())
print(f"Linhas duplicadas: {n_dup} (das quais {n_dup_fraude} são fraudes)")

RESULTADOS["duplicatas_total"] = n_dup
RESULTADOS["duplicatas_fraude"] = n_dup_fraude

# Remoção ANTES da divisão treino/teste, para evitar que a mesma transação
# apareça nos dois conjuntos (vazamento de dados).
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("Dimensões após remoção:", df.shape)
RESULTADOS["n_linhas_sem_duplicatas"] = int(df.shape[0])

## 4. Distribuição da variável-alvo (desbalanceamento)

In [ ]:
contagem = df["Class"].value_counts().sort_index()
pct_fraude = 100 * contagem[1] / contagem.sum()
razao = contagem[0] / contagem[1]
print(contagem)
print(f"Fraudes: {pct_fraude:.3f}% | razão legítimas/fraudes ≈ {razao:.0f}:1")

RESULTADOS["n_legitimas"] = int(contagem[0])
RESULTADOS["n_fraudes"] = int(contagem[1])
RESULTADOS["pct_fraudes"] = round(float(pct_fraude), 4)
RESULTADOS["razao_desbalanceamento"] = round(float(razao), 1)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["Legítima (0)", "Fraude (1)"], contagem.values, color=["#4C72B0", "#C44E52"])
ax.set_yscale("log")
ax.set_ylabel("Número de transações (escala log)")
ax.set_title("Distribuição das classes")
for i, v in enumerate(contagem.values):
    ax.text(i, v * 1.15, f"{v:,}".replace(",", "."), ha="center")
salvar_fig("fig01_distribuicao_classes.png")

## 5. Atributo `Amount` (valor da transação)

In [ ]:
resumo_amount = df.groupby("Class")["Amount"].describe()
print(resumo_amount)
for c, nome in [(0, "legit"), (1, "fraude")]:
    s = df.loc[df["Class"] == c, "Amount"]
    RESULTADOS[f"amount_media_{nome}"] = round(float(s.mean()), 2)
    RESULTADOS[f"amount_mediana_{nome}"] = round(float(s.median()), 2)
    RESULTADOS[f"amount_max_{nome}"] = round(float(s.max()), 2)
    RESULTADOS[f"amount_pct_zero_{nome}"] = round(100 * float((s == 0).mean()), 3)

# Outliers pelo critério do IQR (1,5 x IQR) no conjunto completo
q1, q3 = df["Amount"].quantile([0.25, 0.75])
lim_sup = q3 + 1.5 * (q3 - q1)
RESULTADOS["amount_pct_outliers_iqr"] = round(100 * float((df["Amount"] > lim_sup).mean()), 2)
print(f"Transações acima de Q3 + 1,5·IQR: {RESULTADOS['amount_pct_outliers_iqr']}%")

# Teste KS: as distribuições de Amount diferem entre classes?
ks_amt = ks_2samp(df.loc[df.Class == 0, "Amount"], df.loc[df.Class == 1, "Amount"])
RESULTADOS["amount_ks"] = round(float(ks_amt.statistic), 3)

fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(0, np.log1p(df["Amount"].max()), 50)
ax.hist(np.log1p(df.loc[df.Class == 0, "Amount"]), bins=bins, density=True, alpha=0.6, label="Legítima")
ax.hist(np.log1p(df.loc[df.Class == 1, "Amount"]), bins=bins, density=True, alpha=0.6, label="Fraude", color="#C44E52")
ax.set_xlabel("log(1 + Amount)")
ax.set_ylabel("Densidade")
ax.set_title("Distribuição do valor da transação por classe")
ax.legend()
salvar_fig("fig02_amount_por_classe.png")

## 6. Atributo `Time`

`Time` é o número de segundos desde a **primeira** transação do dataset. O horário do relógio em que a coleta começou não é informado, então a hora derivada aqui é **relativa ao início da coleta**, não o horário real do dia.

In [ ]:
df["hora_relativa"] = ((df["Time"] // 3600) % 24).astype(int)

por_hora = df.groupby("hora_relativa")["Class"].agg(transacoes="count", fraudes="sum")
por_hora["taxa_fraude_pct"] = 100 * por_hora["fraudes"] / por_hora["transacoes"]
print(por_hora)

RESULTADOS["hora_menor_volume"] = int(por_hora["transacoes"].idxmin())
RESULTADOS["hora_maior_taxa_fraude"] = int(por_hora["taxa_fraude_pct"].idxmax())
RESULTADOS["maior_taxa_fraude_hora_pct"] = round(float(por_hora["taxa_fraude_pct"].max()), 3)
RESULTADOS["taxa_fraude_media_hora_pct"] = round(float(por_hora["taxa_fraude_pct"].mean()), 3)

fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.bar(por_hora.index, por_hora["transacoes"], color="#4C72B0", alpha=0.6, label="Transações")
ax1.set_xlabel("Hora relativa ao início da coleta")
ax1.set_ylabel("Transações")
ax2 = ax1.twinx()
ax2.plot(por_hora.index, por_hora["taxa_fraude_pct"], color="#C44E52", marker="o", label="Taxa de fraude (%)")
ax2.set_ylabel("Taxa de fraude (%)")
ax1.set_title("Volume de transações e taxa de fraude por hora")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=2)
salvar_fig("fig03_tempo_hora.png")

## 7. Componentes PCA (V1–V28): correlação e separabilidade entre classes

In [ ]:
cols_v = [f"V{i}" for i in range(1, 29)]

# Correlação de Pearson de cada atributo com a classe (ponto-bisserial)
corr_classe = df[cols_v + ["Amount", "Time"]].corrwith(df["Class"]).sort_values(key=np.abs, ascending=False)

# Estatística KS entre as distribuições de cada atributo nas duas classes
ks = {c: ks_2samp(df.loc[df.Class == 0, c], df.loc[df.Class == 1, c]).statistic for c in cols_v}
ks = pd.Series(ks).sort_values(ascending=False)

tabela_sep = pd.DataFrame({"corr_com_classe": corr_classe.reindex(ks.index), "KS": ks})
print(tabela_sep.head(10).round(3))

RESULTADOS["top10_ks"] = {k: round(float(v), 3) for k, v in ks.head(10).items()}
RESULTADOS["top10_corr_classe"] = {k: round(float(v), 3) for k, v in corr_classe.head(10).items()}
RESULTADOS["n_V_com_ks_menor_0_1"] = int((ks < 0.1).sum())

# Correlação entre os próprios componentes (PCA gera componentes ortogonais)
corr_v = df[cols_v].corr().to_numpy(copy=True)
np.fill_diagonal(corr_v, 0)
RESULTADOS["max_corr_abs_entre_V"] = round(float(np.abs(corr_v).max()), 3)

fig, ax = plt.subplots(figsize=(6, 4))
top = corr_classe.head(12)
ax.barh(top.index[::-1], top.values[::-1], color=["#C44E52" if v < 0 else "#4C72B0" for v in top.values[::-1]])
ax.set_xlabel("Correlação com Class")
ax.set_title("Atributos mais correlacionados com a fraude")
salvar_fig("fig04_correlacao_classe.png")

top4 = list(ks.head(4).index)
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, c in zip(axes, top4):
    lo, hi = df[c].quantile([0.001, 0.999])
    b = np.linspace(lo, hi, 50)
    ax.hist(df.loc[df.Class == 0, c].clip(lo, hi), bins=b, density=True, alpha=0.6, label="Legítima")
    ax.hist(df.loc[df.Class == 1, c].clip(lo, hi), bins=b, density=True, alpha=0.6, label="Fraude", color="#C44E52")
    ax.set_title(f"{c} (KS = {ks[c]:.2f})")
axes[0].legend(fontsize=8)
salvar_fig("fig05_top4_ks.png")

## 8. Preparação dos dados

Decisões:
1. **Engenharia de atributos:** `Time` (segundos acumulados, sem significado fora desta coleta) é substituído pela hora relativa em codificação cíclica (`hora_sin`, `hora_cos`); `Amount` é transformado em `log_amount = log(1 + Amount)` para reduzir a assimetria.
2. **Divisão estratificada** 80/20 (treino/teste), preservando a proporção de fraudes, com semente fixa.
3. **Padronização robusta** (`RobustScaler`, baseada em mediana e IQR, menos sensível a outliers), ajustada **somente no treino**, dentro do pipeline.
4. **SMOTE** aplicado **somente no treino**, dentro do pipeline (e dentro de cada dobra na validação cruzada), para não contaminar o conjunto de teste com exemplos sintéticos.

In [ ]:
df["hora_sin"] = np.sin(2 * np.pi * df["hora_relativa"] / 24)
df["hora_cos"] = np.cos(2 * np.pi * df["hora_relativa"] / 24)
df["log_amount"] = np.log1p(df["Amount"])

FEATURES = cols_v + ["log_amount", "hora_sin", "hora_cos"]
X = df[FEATURES]
y = df["Class"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print("Treino:", X_tr.shape, "fraudes:", int(y_tr.sum()))
print("Teste :", X_te.shape, "fraudes:", int(y_te.sum()))

RESULTADOS["n_features"] = len(FEATURES)
RESULTADOS["treino_n"], RESULTADOS["treino_fraudes"] = int(len(y_tr)), int(y_tr.sum())
RESULTADOS["teste_n"], RESULTADOS["teste_fraudes"] = int(len(y_te)), int(y_te.sum())

## 9. Resultados parciais: modelos de referência (baselines)

Os baselines servem de ponto de comparação para os modelos da N2 (Random Forest ajustado, XGBoost e SVM). A *accuracy* é reportada apenas para mostrar que ela é enganosa neste problema; as métricas principais são **Recall**, **Precisão**, **F1**, **MCC** e **PR-AUC** (área sob a curva Precisão–Recall).

In [ ]:
modelos = {
    "Dummy (classe majoritária)": Pipeline([("sc", RobustScaler()),
                                            ("clf", DummyClassifier(strategy="most_frequent"))]),
    "Regressão Logística (pesos balanceados)": Pipeline([("sc", RobustScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))]),
    "Regressão Logística + SMOTE": ImbPipeline([("sc", RobustScaler()),
        ("smote", SMOTE(random_state=SEED)),
        ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    "Random Forest (pesos balanceados)": Pipeline([("sc", RobustScaler()),
        ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample",
                                       min_samples_leaf=2, n_jobs=-1, random_state=SEED))]),
}

def avaliar(modelo, X_te, y_te, limiar=0.5):
    proba = modelo.predict_proba(X_te)[:, 1]
    pred = (proba >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_te, pred),
        "precisao": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
        "pr_auc": average_precision_score(y_te, proba),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }, proba

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
tabela, probas, ajustados = {}, {}, {}
for nome, modelo in modelos.items():
    t0 = time.time()
    cv = cross_val_score(modelo, X_tr, y_tr, cv=skf, scoring="average_precision", n_jobs=1)
    modelo.fit(X_tr, y_tr)
    met, proba = avaliar(modelo, X_te, y_te)
    met["cv_pr_auc_media"], met["cv_pr_auc_dp"] = cv.mean(), cv.std()
    met["tempo_s"] = time.time() - t0
    tabela[nome], probas[nome], ajustados[nome] = met, proba, modelo
    print(f"{nome}: PR-AUC teste = {met['pr_auc']:.3f} | CV = {cv.mean():.3f} ± {cv.std():.3f} | {met['tempo_s']:.0f}s")

tab = pd.DataFrame(tabela).T
cols_show = ["accuracy", "precisao", "recall", "f1", "mcc", "roc_auc", "pr_auc",
             "cv_pr_auc_media", "cv_pr_auc_dp", "TN", "FP", "FN", "TP"]
RESULTADOS["baselines"] = {m: {k: (round(float(v), 4) if isinstance(v, float) else v)
                              for k, v in tabela[m].items()} for m in tabela}
tab[cols_show].round(4)

In [ ]:
# Curvas Precisão–Recall no conjunto de teste
fig, ax = plt.subplots(figsize=(5.5, 4))
for nome, proba in probas.items():
    if nome.startswith("Dummy"):
        continue
    p, r, _ = precision_recall_curve(y_te, proba)
    ax.plot(r, p, label=f"{nome} (AP = {average_precision_score(y_te, proba):.3f})")
ax.axhline(y_te.mean(), ls="--", c="gray", label=f"Aleatório (AP = {y_te.mean():.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precisão")
ax.set_title("Curvas Precisão–Recall (teste)")
ax.legend(fontsize=7, loc="lower left")
salvar_fig("fig06_curvas_pr.png")

# Matrizes de confusão dos três modelos não triviais
nomes = [n for n in tabela if not n.startswith("Dummy")]
fig, axes = plt.subplots(1, len(nomes), figsize=(4 * len(nomes), 3.4))
for ax, nome in zip(axes, nomes):
    m = np.array([[tabela[nome]["TN"], tabela[nome]["FP"]], [tabela[nome]["FN"], tabela[nome]["TP"]]])
    ax.imshow(m, cmap="Blues", norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=m.max()))
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{m[i, j]:,}".replace(",", "."), ha="center", va="center",
                    color="white" if m[i, j] > m.max() / 10 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Prev. legítima", "Prev. fraude"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Legítima", "Fraude"])
    ax.set_title(nome, fontsize=9)
salvar_fig("fig07_matrizes_confusao.png")

In [ ]:
# Importância dos atributos (redução média de impureza) na Random Forest
rf = ajustados["Random Forest (pesos balanceados)"].named_steps["clf"]
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
RESULTADOS["rf_top10_importancia"] = {k: round(float(v), 4) for k, v in imp.head(10).items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(imp.head(12).index[::-1], imp.head(12).values[::-1], color="#55A868")
ax.set_xlabel("Importância (redução média de impureza)")
ax.set_title("Random Forest: atributos mais importantes")
salvar_fig("fig08_importancia_rf.png")

## 10. Verificação com divisão temporal

Na prática, o modelo é treinado com transações passadas e aplicado a transações futuras. Para verificar se a divisão aleatória superestima o desempenho, treinamos com as primeiras 80% das transações (ordenadas por `Time`) e testamos nas 20% finais.

In [ ]:
ordem = df["Time"].argsort().values
corte = int(0.8 * len(ordem))
idx_tr, idx_te = ordem[:corte], ordem[corte:]
Xt_tr, Xt_te = X.iloc[idx_tr], X.iloc[idx_te]
yt_tr, yt_te = y.iloc[idx_tr], y.iloc[idx_te]
print("Fraudes treino/teste temporal:", int(yt_tr.sum()), int(yt_te.sum()))
RESULTADOS["temporal_teste_fraudes"] = int(yt_te.sum())

temporal = {}
for nome in ["Regressão Logística (pesos balanceados)", "Random Forest (pesos balanceados)"]:
    from sklearn.base import clone
    m = clone(modelos[nome]).fit(Xt_tr, yt_tr)
    met, _ = avaliar(m, Xt_te, yt_te)
    temporal[nome] = {k: (round(float(v), 4) if isinstance(v, float) else v) for k, v in met.items()}
    print(f"{nome}: PR-AUC temporal = {met['pr_auc']:.3f} (aleatória: {tabela[nome]['pr_auc']:.3f})")
RESULTADOS["temporal"] = temporal

## 11. Resumo para o artigo

Execute a célula abaixo e **copie toda a saída** (e envie o arquivo `figuras.zip`).

In [ ]:
with open("resultados_parciais.json", "w", encoding="utf-8") as f:
    json.dump(RESULTADOS, f, ensure_ascii=False, indent=2)
shutil.make_archive("figuras", "zip", FIG_DIR)

print(json.dumps(RESULTADOS, ensure_ascii=False, indent=2))
print("\nFiguras geradas:", sorted(os.listdir(FIG_DIR)))

try:
    from google.colab import files
    files.download("figuras.zip")
    files.download("resultados_parciais.json")
except Exception:
    print("Fora do Colab: os arquivos estão na pasta atual.")